# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') 

### Example 1

#### Create our QandA application

In [4]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.llms import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.chains import LLMChain


In [6]:
file = './data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [9]:
# !pip install --upgrade --force-reinstall sentence-transformers

In [9]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader])

c:\Users\Nada\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [10]:
llm = ChatOpenAI(temperature = 0.0)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [11]:
data[5]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 5}, page_content=": 5\nname: Smooth Comfort Check Shirt, Slightly Fitted\ndescription: Our men's slightly fitted check shirt is the perfect choice for your wardrobe! Customers love how it fits right out of the dryer. Size & Fit: Slightly Fitted, Relaxed through the chest and sleeve with a slightly slimmer waist. Fabric & Care: 100% cotton poplin, with wrinkle-free performance that won't wash out. Our innovative TrueCoolآ® fabric wicks moisture away from your skin and helps it dry quickly. Additional Features: Traditional styling with a button-down collar and a single patch pocket. Imported.")

In [12]:
data[6]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 6}, page_content=': 6\nname: Mens Outdoor Research Apex Gloves\ndescription: These Outdoor Research Arete gloves are perfect for your backcountry adventures. They feature a tough Taslan nylon shell and removable Gore-Texآ® nylon fleece liner that is quick-drying and provides complete protection and warmth. The stretchy nylon Lycra elastane soft-shell fabric on the back of the hands allows for freedom of movement, while the ergonomic design lets your hands move naturally for maximum comfort. The gauntlet closure seals out snow and cold wind, and a special pocket allows for a heat pack to be inserted to help keep your hands warm. Both the shell and 100-weight-fleece liner can be worn alone or together for maximum warmth. Imported. These gloves also offer serious waterproof protection from the elements, and are highly breathable to allow moisture vapor to escape. Get the durability you expect from Gore-Tex fabri

#### Hard-coded examples

In [13]:
from langchain.prompts import PromptTemplate

In [14]:
from langchain.prompts import PromptTemplate
from langchain.schema import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Does the Smooth Comfort Check Shirt have wrinkle-free fabric?",
        "answer": "Yes, it is made with wrinkle-free cotton poplin fabric that won't wash out."
    },
    {
        "query": "Are the Outdoor Research Apex Gloves waterproof?",
        "answer": "Yes, they provide serious waterproof protection using Gore-Tex fabric."
    }
]
# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Does the Smooth Comfort Check Shirt have wrinkle-free fabric?\n"
             "   Answer: Yes, it is made with wrinkle-free cotton poplin fabric that won't wash out.\n"
             "2. Query: Are the Outdoor Research Apex Gloves waterproof?\n"
             "   Answer: Yes, they provide serious waterproof protection using Gore-Tex fabric.\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI()

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Smooth Comfort Check Shirt moisture-wicking?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


C:\Users\Nada\AppData\Local\Temp\ipykernel_11592\2422589220.py:43: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
C:\Users\Nada\AppData\Local\Temp\ipykernel_11592\2422589220.py:53: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = llm_chain.run({"query": query})


answer='Yes, the Smooth Comfort Check Shirt is made with moisture-wicking fabric to keep you cool and dry all day.'


#### LLM-Generated examples

In [15]:
from langchain.evaluation.qa import QAGenerateChain

In [16]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

In [17]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [18]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

c:\Users\Nada\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain\chains\llm.py:369: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [19]:
new_examples[0]

{'qa_pairs': {'query': "According to the document, what materials are the Women's Campside Oxfords made of for a broken-in feel and look?",
  'answer': "The Women's Campside Oxfords are made of soft canvas material for a broken-in feel and look."}}

In [20]:
data[0]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXTآ® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [21]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "According to the document, what materials are the Women's Campside Oxfords made of for a broken-in feel and look?",
  'answer': "The Women's Campside Oxfords are made of soft canvas material for a broken-in feel and look."},
 {'query': 'What are the dimensions of the small and medium sizes for the Recycled Waterhog Dog Mat?',
  'answer': 'The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What are some key features of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': 'The key features of the swimsuit include bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant fabric, UPF 50+ rated fabric for sun protection, crossover no-slip straps, fully lined bottom for secure fit and maximum coverage, and it is recommended to be machine washed and line dried for best results.'},
 {'query': 'What is the fabric composition of the Refr

#### Combine examples

In [22]:
# examples += new_example
examples += d_flattened

In [23]:
examples[0]

{'query': 'Does the Smooth Comfort Check Shirt have wrinkle-free fabric?',
 'answer': "Yes, it is made with wrinkle-free cotton poplin fabric that won't wash out."}

In [24]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Does the Smooth Comfort Check Shirt have wrinkle-free fabric?',
 'result': "Yes, the Smooth Comfort Check Shirt is made of 100% cotton poplin with wrinkle-free performance that won't wash out."}

### Manual Evaluation - Fun part

In [25]:
import langchain
langchain.debug = True

In [26]:
qa.invoke(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Does the Smooth Comfort Check Shirt have wrinkle-free fabric?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Does the Smooth Comfort Check Shirt have wrinkle-free fabric?",
  "context": ": 155\nname: Careless Indestructible Tee, Timeless Fit, Long-Sleeve Henley\ndescription: Made of soft cotton that resists wrinkles, stains, shrinking, fading and pilling, our resilient henley keeps its shape wash after wash. Size & Fit Traditional Fit: Relaxed through the chest, sleeve and waist.\n\nWhy We Love It Weâ€™ve put our Unshrinkable Carefree Tees to the test, in the lab and in the field. And weâ€™re glad to say our unshrinkable t-shirts passed with flying colors. Whatâ€™s more, the 100% cotton fabric resists wrinkles, stains, fading,

{'query': 'Does the Smooth Comfort Check Shirt have wrinkle-free fabric?',
 'result': "Yes, the Smooth Comfort Check Shirt is made of 100% cotton poplin with wrinkle-free performance that won't wash out."}

In [27]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [28]:
examples += d_flattened

In [29]:
examples

[{'query': 'Does the Smooth Comfort Check Shirt have wrinkle-free fabric?',
  'answer': "Yes, it is made with wrinkle-free cotton poplin fabric that won't wash out."},
 {'query': 'Are the Outdoor Research Apex Gloves waterproof?',
  'answer': 'Yes, they provide serious waterproof protection using Gore-Tex fabric.'},
 {'query': "According to the document, what materials are the Women's Campside Oxfords made of for a broken-in feel and look?",
  'answer': "The Women's Campside Oxfords are made of soft canvas material for a broken-in feel and look."},
 {'query': 'What are the dimensions of the small and medium sizes for the Recycled Waterhog Dog Mat?',
  'answer': 'The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What are some key features of the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece as described in the document?",
  'answer': 'The key features of the swimsuit include bright colors, ruffles, exclusive whim

In [30]:
predictions = qa.batch(examples)



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [31]:
predictions

[{'query': 'Does the Smooth Comfort Check Shirt have wrinkle-free fabric?',
  'answer': "Yes, it is made with wrinkle-free cotton poplin fabric that won't wash out.",
  'result': "Yes, the Smooth Comfort Check Shirt is made of 100% cotton poplin with wrinkle-free performance that won't wash out."},
 {'query': 'Are the Outdoor Research Apex Gloves waterproof?',
  'answer': 'Yes, they provide serious waterproof protection using Gore-Tex fabric.',
  'result': 'Yes, the Outdoor Research Apex Gloves offer serious waterproof protection from the elements, as they are made with a tough Taslan nylon shell and a removable Gore-Tex nylon fleece liner that provides complete protection and warmth.'},
 {'query': "According to the document, what materials are the Women's Campside Oxfords made of for a broken-in feel and look?",
  'answer': "The Women's Campside Oxfords are made of soft canvas material for a broken-in feel and look.",
  'result': "The Women's Campside Oxfords are made of soft canvas m

In [32]:
from langchain.evaluation.qa import QAEvalChain

In [33]:
llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [34]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [35]:
graded_outputs

[{'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'}]

In [36]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    # print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Does the Smooth Comfort Check Shirt have wrinkle-free fabric?
Real Answer: Yes, it is made with wrinkle-free cotton poplin fabric that won't wash out.
Predicted Answer: Yes, the Smooth Comfort Check Shirt is made of 100% cotton poplin with wrinkle-free performance that won't wash out.

Example 1:
Question: Are the Outdoor Research Apex Gloves waterproof?
Real Answer: Yes, they provide serious waterproof protection using Gore-Tex fabric.
Predicted Answer: Yes, the Outdoor Research Apex Gloves offer serious waterproof protection from the elements, as they are made with a tough Taslan nylon shell and a removable Gore-Tex nylon fleece liner that provides complete protection and warmth.

Example 2:
Question: According to the document, what materials are the Women's Campside Oxfords made of for a broken-in feel and look?
Real Answer: The Women's Campside Oxfords are made of soft canvas material for a broken-in feel and look.
Predicted Answer: The Women's Campside Oxfords

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [87]:
from langchain_huggingface import HuggingFaceEmbeddings
loader = TextLoader("./data/nyc_text.txt")
index = VectorstoreIndexCreator(embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})).from_loaders([loader])


llm = ChatOpenAI(temperature= 0)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=index.vectorstore.as_retriever(),
    return_source_documents=True,
)

c:\Users\Nada\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain\indexes\vectorstore.py:171: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  warnings.warn(


In [88]:
# testing it out

question = "How is Manhattan geographically divided into different regions?"
result = qa_chain.invoke({"query": question})
result["result"]

'Manhattan Island is loosely divided into the Lower, Midtown, and Uptown regions. Uptown Manhattan is further divided by Central Park into the Upper East Side and the Upper West Side, with Harlem located above the park. Harlem borders the Bronx and was historically occupied by Jewish and Italian Americans before the Great Migration.'

In [89]:
result

{'query': 'How is Manhattan geographically divided into different regions?',
 'result': 'Manhattan Island is loosely divided into the Lower, Midtown, and Uptown regions. Uptown Manhattan is further divided by Central Park into the Upper East Side and the Upper West Side, with Harlem located above the park. Harlem borders the Bronx and was historically occupied by Jewish and Italian Americans before the Great Migration.',
 'source_documents': [Document(id='538244c9-7b57-44d5-b822-94f31d5586dc', metadata={'source': './data/nyc_text.txt'}, page_content='Manhattan Island is loosely divided into the Lower, Midtown, and Uptown regions. Uptown Manhattan is divided by Central Park into the Upper East Side and the Upper West Side, and above the park is Harlem, bordering the Bronx (Bronx County).\nHarlem was predominantly occupied by Jewish and Italian Americans in the 19th century until the Great Migration. It was the center of the Harlem Renaissance.'),
  Document(id='c247540e-d2bf-4cf9-b6c2-b

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [153]:
eval_questions = [
    "What is Brooklyn known for?",
    "Which borough is the most ethnically diverse in NYC?",
    "What is the Bronx famous for?",
    "How is Staten Island connected to other boroughs?",
    "What architectural style is the Chrysler Building known for?"
]

eval_answers = [
    "Cultural diversity and Coney Island.",
    "Queens",
    "The birthplace of hip hop and Yankee Stadium.",
    "By the Verrazzano-Narrows Bridge and Staten Island Ferry.",
    "Art Deco."
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [91]:
examples

[{'query': 'What is Brooklyn known for?',
  'ground_truths': ['Cultural diversity and Coney Island.']},
 {'query': 'Which borough is the most ethnically diverse in NYC?',
  'ground_truths': ['Queens']},
 {'query': 'What is the Bronx famous for?',
  'ground_truths': ['The birthplace of hip hop and Yankee Stadium.']},
 {'query': 'How is Staten Island connected to other boroughs?',
  'ground_truths': ['By the Verrazzano-Narrows Bridge and Staten Island Ferry.']},
 {'query': 'What architectural style is the Chrysler Building known for?',
  'ground_truths': ['Art Deco.']}]

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [92]:
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

'The borough of Queens is the most ethnically diverse in New York City. It is home to the largest Asian American population in the state, as well as the largest Andean populations in the United States. Additionally, Queens is considered the most ethnically and linguistically diverse urban area in the world.'

In [93]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]


In [94]:
result_updated

{'question': 'Which borough is the most ethnically diverse in NYC?',
 'answer': 'The borough of Queens is the most ethnically diverse in New York City. It is home to the largest Asian American population in the state, as well as the largest Andean populations in the United States. Additionally, Queens is considered the most ethnically and linguistically diverse urban area in the world.',
 'contexts': [Document(id='f56b64d2-0d63-4dc5-bcd4-9068ab6cbe24', metadata={'source': './data/nyc_text.txt'}, page_content="descent both stood at about 20,000 each, while people of Czech, Lithuanian, Portuguese, Scotch-Irish, and Welsh descent all numbered between 12,000 and 14,000. Arab Americans number over 160,000 in New York City, with the highest concentration in Brooklyn. Central Asians, primarily Uzbek Americans, are a rapidly growing segment of the city's non-Hispanic White population, enumerating over 30,000, and including more than half of all Central Asian immigrants to the United States, mo

In [47]:
pip install --no-cache-dir recordclass

  Obtaining dependency information for recordclass from https://files.pythonhosted.org/packages/57/57/e7c61eff0271bfb0c343eba2b3d799eb3c1544b622d9f9b31dc19bc78fcb/recordclass-0.23.1-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/44.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/44.6 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.6 kB ? eta -:--:--
     ----------------- -------------------- 20.5/44.6 kB 131.3 kB/s eta 0:00:01
     ----------------- -------------------- 20.5/44.6 kB 131.3 kB/s eta 0:00:01
     -------------------------- ----------- 30.7/44.6 kB 145.2 kB/s eta 0:00:01
     -------------------------------------- 44.6/44.6 kB 169.0 kB/s eta 0:00:00
   ---------------------------------------- 0.0/249.2 kB ? eta -:--:--
   - -------------------------------------- 10.2/249.2 kB ? eta -:--:--
   ------ -------------------------------- 41.0/249.2 kB 653.6 kB/s eta 0:00:01
   ------------------- 


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
pip install ragas==0.1.9

  Obtaining dependency information for ragas==0.1.9 from https://files.pythonhosted.org/packages/9e/55/09fa95c22502322e71a90194712ea23c7381a52c7c7e10ec9215c4d71b93/ragas-0.1.9-py3-none-any.whl.metadata
  Obtaining dependency information for pysbd>=0.3.4 from https://files.pythonhosted.org/packages/48/0a/c99fb7d7e176f8b176ef19704a32e6a9c6aafdf19ef75a187f701fc15801/pysbd-0.3.4-py3-none-any.whl.metadata
  Obtaining dependency information for appdirs from https://files.pythonhosted.org/packages/3b/00/2344469e2084fb287c2e0b57b72910309874c3245463acd6cf5e3db69324/appdirs-1.4.4-py2.py3-none-any.whl.metadata
   ---------------------------------------- 0.0/86.1 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/86.1 kB ? eta -:--:--
   ------------------- -------------------- 41.0/86.1 kB 487.6 kB/s eta 0:00:01
   ---------------------------------------- 86.1/86.1 kB 689.9 kB/s eta 0:00:00
   ---------------------------------------- 0.0/71.1 kB ? eta -:--:--
   -------------------


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [95]:
from ragas.integrations.langchain import EvaluatorChain 
# from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_recall,
)

# create evaluation chains
faithfulness_chain   = EvaluatorChain(metric=faithfulness)
answer_rel_chain     = EvaluatorChain(metric=answer_relevancy)
context_rel_chain    = EvaluatorChain(metric=context_relevancy)
context_recall_chain = EvaluatorChain(metric=context_recall)

1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [96]:
# Recheck the result that we are going to validate.
result

{'query': 'Which borough is the most ethnically diverse in NYC?',
 'result': 'The borough of Queens is the most ethnically diverse in New York City. It is home to the largest Asian American population in the state, as well as the largest Andean populations in the United States. Additionally, Queens is considered the most ethnically and linguistically diverse urban area in the world.',
 'source_documents': [Document(id='f56b64d2-0d63-4dc5-bcd4-9068ab6cbe24', metadata={'source': './data/nyc_text.txt'}, page_content="descent both stood at about 20,000 each, while people of Czech, Lithuanian, Portuguese, Scotch-Irish, and Welsh descent all numbered between 12,000 and 14,000. Arab Americans number over 160,000 in New York City, with the highest concentration in Brooklyn. Central Asians, primarily Uzbek Americans, are a rapidly growing segment of the city's non-Hispanic White population, enumerating over 30,000, and including more than half of all Central Asian immigrants to the United State

**Faithfulness**

In [99]:
result_updated = {
    "question": result["query"],
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]]
}

In [103]:
eval_result = await faithfulness_chain.ainvoke(result_updated)
print("Faithfulness Score:", eval_result["faithfulness"])

Faithfulness Score: 1.0


In [102]:
eval_result

{'question': 'Which borough is the most ethnically diverse in NYC?',
 'answer': 'The borough of Queens is the most ethnically diverse in New York City. It is home to the largest Asian American population in the state, as well as the largest Andean populations in the United States. Additionally, Queens is considered the most ethnically and linguistically diverse urban area in the world.',
 'contexts': ["descent both stood at about 20,000 each, while people of Czech, Lithuanian, Portuguese, Scotch-Irish, and Welsh descent all numbered between 12,000 and 14,000. Arab Americans number over 160,000 in New York City, with the highest concentration in Brooklyn. Central Asians, primarily Uzbek Americans, are a rapidly growing segment of the city's non-Hispanic White population, enumerating over 30,000, and including more than half of all Central Asian immigrants to the United States, most settling in Queens or Brooklyn. Albanian Americans are most highly concentrated in the Bronx, while Astori

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [107]:
fake_result = result_updated.copy()
fake_result["result_updated"] = "we are the champions"
eval_result = await faithfulness_chain.ainvoke(fake_result)
eval_result["faithfulness"]

1.0

**Context Relevancy**

In [118]:
result_updated_context = {
    "question": result["query"],
    "answer": result["result"],
    "ground_truth": "Queens",
    "contexts": [doc.page_content for doc in result["source_documents"]]
}

In [119]:
eval_result = await context_recall_chain.ainvoke(result_updated_context)
eval_result["context_recall"]

1.0

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [120]:
from langchain.schema import Document
fake_result = result_updated_context.copy()
fake_result["source_documents"] = [Document(page_content="I love christmas")]
eval_result = await context_recall_chain.ainvoke(fake_result)
eval_result["context_recall"]

1.0

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [154]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness

# Step 1: Get predictions
predictions = qa_chain.batch(examples)

# Step 2: Format data
formatted_data = []
for i, example in enumerate(examples):
    prediction = predictions[i] if i < len(predictions) else None
    if prediction:
        item = {
            "question": example["query"],
            "answer": prediction["result"],
            "contexts": [doc.page_content for doc in prediction["source_documents"]] if "source_documents" in prediction else [],
            "ground_truth": example["ground_truths"][0]
        }
        formatted_data.append(item)



In [155]:
# Convert to Hugging Face Dataset
dataset = Dataset.from_list(formatted_data)

# Evaluate
print("evaluating...")
results = evaluate(dataset, metrics=[faithfulness])
results

evaluating...


Evaluating: 100%|██████████| 5/5 [00:10<00:00,  2.05s/it]


{'faithfulness': 1.0000}

In [156]:
# Evaluate
print("evaluating...")
results = evaluate(dataset, metrics=[context_recall])
results

evaluating...


Evaluating: 100%|██████████| 5/5 [00:02<00:00,  1.69it/s]


{'context_recall': 0.8833}

In [158]:
# Evaluate
print("evaluating...")
results = evaluate(dataset, metrics=[context_recall,faithfulness,answer_relevancy,context_relevancy]),
results

evaluating...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 20/20 [00:17<00:00,  1.17it/s]


({'context_recall': 0.9333, 'faithfulness': 0.9833, 'answer_relevancy': 0.9802, 'context_relevancy': 0.0676},)